# ChordSense Pro — Training on Colab
GPU: L4 24GB | Runtime: GPU

## ⚙️ GPU Config — chỉnh tại đây trước khi train

In [ ]:
import os

# --- Tuỳ chỉnh theo GPU đang dùng ---
# T4  16GB  → BATCH_SIZE=16
# L4  24GB  → BATCH_SIZE=32
# A100 40GB → BATCH_SIZE=64
os.environ['BATCH_SIZE']  = '32'    # ← đổi ở đây nếu đổi GPU
os.environ['NUM_WORKERS'] = '4'
os.environ['EPOCHS']      = '20'
os.environ['LR_HEAD']     = '2e-4'
os.environ['SEED']        = '42'
os.environ['FP16']        = 'true'

print('Config overrides set.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!nvidia-smi

In [ ]:
DRIVE_PROJECT = "/content/drive/MyDrive/ChordSensePro"
REPO_DIR = "/content/ChordSensePro"

if not os.path.exists(REPO_DIR):
    !git clone --branch qkhenh https://github.com/qkhenh/ChordSensePro.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull origin qkhenh

os.chdir(os.path.join(REPO_DIR, "AI"))
!pwd

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
DATA_SRC = os.path.join(DRIVE_PROJECT, "DE", "data", "export", "v2")
DATA_V2 = "/content/data_v2"
DATA_DST = "/content/data_v3"

if not os.path.exists(DATA_V2):
    !cp -r {DATA_SRC} {DATA_V2}
    print("Data v2 copied.")
else:
    print("Data v2 already exists.")

# Build data_v3 (adds bass_id + hardlinks audio in ~3s)
!python ../DE/scripts/export_dataset_v3.py --v2-dir {DATA_V2} --out {DATA_DST}
!ls {DATA_DST}

In [ ]:
import yaml
from pathlib import Path

# Chọn config muốn chạy: v3_5s_p1 | v3_3s_p1
RUN_NAME = "v3_5s_p1"
cfg_path = Path(f"configs/{RUN_NAME}.yaml")
cfg = yaml.safe_load(cfg_path.read_text())

cfg["data_dir"] = DATA_DST
cfg["checkpoint_dir"] = os.path.join(DRIVE_PROJECT, "AI", "results", RUN_NAME)
cfg["results_csv"] = os.path.join(DRIVE_PROJECT, "AI", "results", RUN_NAME, "results.csv")

colab_cfg_path = f"configs/colab_{RUN_NAME}.yaml"
with open(colab_cfg_path, "w") as f:
    yaml.dump(cfg, f, default_flow_style=False)

print("Config saved to", colab_cfg_path)

In [ ]:
!python -m src.train --config {colab_cfg_path}

## Resume from checkpoint

In [ ]:
CKPT = os.path.join(cfg["checkpoint_dir"], "last.pt")
if os.path.exists(CKPT):
    !python -m src.train --config {colab_cfg_path} --resume {CKPT}
else:
    print("No checkpoint found.")

## Extract Embeddings (optional — for baseline experiments)

In [ ]:
!python -m src.extract_embeddings --config configs/colab.yaml --out_dir results/embeddings

## Quick Inference Test

In [ ]:
import numpy as np

test_audio_dir = os.path.join(DATA_DST, "test", "audio")
test_files = sorted(os.listdir(test_audio_dir))[:3]

BEST_CKPT = os.path.join(cfg["checkpoint_dir"], "best.pt")

for f in test_files:
    audio_path = os.path.join(test_audio_dir, f)
    !python -m src.infer --config configs/colab.yaml --checkpoint {BEST_CKPT} --audio {audio_path}

## Export ONNX + Benchmark

In [ ]:
!python -m src.export_onnx --config configs/colab.yaml --checkpoint {BEST_CKPT} --output results/model.onnx --benchmark

---
## 🔥 Phase 2 — LoRA Fine-tune MERT
Chạy sau khi Phase 1 xong. Cần `best.pt` từ Phase 1.

In [ ]:
# Phase 2 config overrides (Chọn v3_5s_p2 hoặc v3_3s_p2)
P2_RUN_NAME = 'v3_5s_p2'    # hoặc 'v3_3s_p2'
P1_RUN_NAME = 'v3_5s_p1'    # hoặc 'v3_3s_p1'
os.environ['BATCH_SIZE']  = '16' if '5s' in P2_RUN_NAME else '24'
os.environ['NUM_WORKERS'] = '4'
os.environ['EPOCHS']      = '25'

import yaml
from pathlib import Path

cfg_p2 = yaml.safe_load(Path(f'configs/{P2_RUN_NAME}.yaml').read_text())
cfg_p2['data_dir'] = DATA_DST
cfg_p2['checkpoint_dir'] = os.path.join(DRIVE_PROJECT, 'AI', 'results', P2_RUN_NAME)
cfg_p2['results_csv'] = os.path.join(DRIVE_PROJECT, 'AI', 'results', P2_RUN_NAME, 'results.csv')

colab_p2_path = f'configs/colab_{P2_RUN_NAME}.yaml'
with open(colab_p2_path, 'w') as f:
    yaml.dump(cfg_p2, f, default_flow_style=False)

P1_BEST = os.path.join(DRIVE_PROJECT, 'AI', 'results', P1_RUN_NAME, 'best.pt')
print('Phase 1 checkpoint:', P1_BEST)
print('Exists:', os.path.exists(P1_BEST))

In [ ]:
# Train Phase 2 — khởi tạo từ Phase 1 checkpoint
!python -m src.train --config {colab_p2_path} --phase1_ckpt {P1_BEST}

In [ ]:
# Resume Phase 2 nếu bị disconnect
P2_LAST = os.path.join(cfg_p2['checkpoint_dir'], 'last.pt')
if os.path.exists(P2_LAST):
    !python -m src.train --config {colab_p2_path} --resume {P2_LAST}
else:
    print('No Phase 2 checkpoint to resume.')

In [ ]:
# Export Phase 2 model to ONNX
P2_BEST = os.path.join(cfg_p2['checkpoint_dir'], 'best.pt')
OUT_ONNX = os.path.join(DRIVE_PROJECT, 'AI', 'results', f'model_{P2_RUN_NAME}.onnx')
!python -m src.export_onnx --config {colab_p2_path} --checkpoint {P2_BEST} --output {OUT_ONNX} --benchmark